<a href="https://www.kaggle.com/code/parsalatifi/what-makes-a-yt-video-go-viral?scriptVersionId=357001944" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<div align="center">


<br/>

<img src="https://img.shields.io/badge/XGBoost-Model-blue?style=flat-square&logo=xgboost" />
<img src="https://img.shields.io/badge/Accuracy-82%25-success?style=flat-square" />
<img src="https://img.shields.io/badge/ROC--AUC-0.89-orange?style=flat-square" />
<img src="https://img.shields.io/badge/F1(viral)-0.64-red?style=flat-square" />

<br/>
<br/>

<h1 align="center">Data Leakage and Class Imbalance in Viral Video Prediction</h1>

<h3 align="center"><em>A case study on honest model evaluation</em></h3>

<br/>

<hr width="60%"/>

</div>

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
plt.style.use("dark_background")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import roc_curve

In [ ]:
data = pd.read_csv('/kaggle/input/datasets/vedauv/what-makes-a-yt-video-go-viral-12000-vids/viral_videos.csv')
data.head()

# EDA 

In [ ]:
# Show some countplots for the categorical features

categorical_features = ['category','upload_day','thumbnail_face']
plt.figure(figsize=(15, 8))
for i, feature in enumerate(categorical_features):
    plt.subplot(2, 2, i + 1)
    ax = sns.countplot(data=data, x=feature)
    plt.title(f'Countplot of {feature}')
    plt.xticks(rotation=45)
    for container in ax.containers:
        ax.bar_label(container, fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols = ['duration_sec', 'title_length', 'upload_hour', 'tags_count',
                'description_length', 'subscriber_count', 'views', 'likes',
                'comments', 'shares', 'viral']

data['viral'] = data['viral'].astype(int)

corr_matrix = data[numeric_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap of Numeric Features', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
ax = sns.countplot(data=data, x='viral', hue='category')
plt.title('Viral vs Non-Viral Videos by Category', fontsize=14)
plt.xlabel('Viral (1) vs Non-Viral (0)', fontsize=12)
plt.ylabel('Count', fontsize=12)

legend = plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
cat_names = [t.get_text() for t in legend.get_texts()]

for container, cat in zip(ax.containers, cat_names):
    labels = [cat for _ in container]
    ax.bar_label(container, labels=labels, fontsize=9, padding=3)

plt.tight_layout()
plt.show()

# Preprocessing

In [ ]:
data.info()

In [ ]:
day_order = ['Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri']
data['upload_day_num'] = data['upload_day'].map({d: i for i, d in enumerate(day_order)})

data['upload_day_sin'] = np.sin(2 * np.pi * data['upload_day_num'] / 7)
data['upload_day_cos'] = np.cos(2 * np.pi * data['upload_day_num'] / 7)

data = data.drop(columns=['upload_day', 'upload_day_num'])

In [ ]:
data = pd.get_dummies(data, columns=['category'], drop_first=False)

In [ ]:
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)
data.drop_duplicates(inplace=True)
data.drop(columns=['video_id'], axis=1, inplace=True)

In [ ]:
# Drop leaky columns: views, likes, comments, shares reveal viral outcome
leak_cols = ['views', 'likes', 'comments', 'shares']
X = data.drop(columns=leak_cols + ['viral'])
y = data['viral']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

print("\nDistribution in y_train:")
print(y_train.value_counts(normalize=True))
print("\nDistribution in y_test:")
print(y_test.value_counts(normalize=True))

# Model

In [ ]:
scale = (y_train == 0).sum() / (y_train == 1).sum()   # ~4

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'AUC = 0.89')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

# Data Leakage and Class Imbalance in Viral Video Prediction

---

## The Setup

I trained an XGBoost classifier to predict whether a video would go viral. The features covered things you'd know *before* publishing: category, duration, title length, upload timing, subscriber count, tags, and description length. Simple enough.

The first run returned an F1-score of **1.00** across both classes. ROC-AUC: **0.9999913**.

That number doesn't happen in the real world.

---

## The Leakage Problem

Perfect scores are almost always a symptom of **data leakage**. In this case, the model wasn't learning to *predict* virality — it was learning to *recognize* the variables that defined it.

The dataset included engagement metrics like `views`, `likes`, `comments`, and `shares`. These aren't inputs. They're outcomes. If a video has 2 million views, of course it's viral. The model figured that out immediately and ignored everything else.

This is a classic mistake in applied ML. The model performs brilliantly on paper and fails completely in deployment because the leaked features won't exist when you actually need the prediction. No one has view counts before a video goes live.

Once I removed the leakage columns:

```python
leak_cols = ['views', 'likes', 'comments', 'shares', 'video_id']
X = data.drop(columns=leak_cols + ['viral'])
```

the model's performance dropped to something honest:

| Metric | With Leakage | Without Leakage |
|---|---|---|
| Accuracy | 1.00 | 0.82 |
| F1 (viral) | 1.00 | 0.64 |
| ROC-AUC | 0.9999 | 0.89 |

The drop was painful but necessary. The first model was useless. The second one might actually work.

---

## The Imbalance Issue

The viral class is rare. Out of 12,000 videos, only **2,400 (20%)** crossed the viral threshold. This imbalance shapes everything about how the model behaves.

With a standard split and no adjustment, the model learned to favor the majority class:

| Class | Precision | Recall | F1 |
|---|---|---|---|
| Non-viral (0) | 0.89 | 0.94 | 0.91 |
| Viral (1) | 0.68 | **0.53** | 0.60 |

It correctly identified 94% of non-viral videos but only caught **53% of the viral ones**. In other words, it missed nearly half of the events I actually cared about.

### The Fix

Adding class weights tells the model to treat errors on the minority class as more costly:

```python
scale = (y_train == 0).sum() / (y_train == 1).sum()
model = XGBClassifier(scale_pos_weight=scale, ...)
```

This shifted the trade-off:

| Class | Precision | Recall | F1 |
|---|---|---|---|
| Non-viral (0) | 0.94 | 0.82 | 0.88 |
| Viral (1) | 0.53 | **0.80** | 0.64 |

Recall for viral videos jumped to **80%**, but precision fell to 0.53. The model now flags more false positives — but it catches more of what matters.

This is the fundamental tension in imbalanced classification. You can't maximize both precision and recall when the classes are skewed. You have to decide **which error hurts more**. In this context, missing a viral video is worse than flagging a non-viral one for review.

---

## What the Numbers Actually Mean

The current performance is realistic:

- **Accuracy:** 0.82
- **F1 (viral):** 0.64
- **ROC-AUC:** 0.89

It reflects genuine signal in the data — subscriber count, category, timing, and content structure do carry information — but it also reflects the inherent difficulty of the task.

> Most videos don't go viral. The ones that do don't announce themselves in advance. If a model could reliably predict virality from metadata alone, every media company would already use it.

The honest result isn't 100%. It's a model that:

- Does better than random
- Catches most viral videos while accepting some false alarms
- Still gets it wrong about one in five times

That's what real predictive modeling looks like.

---

## Key Takeaways

| Lesson | Why It Matters |
|---|---|
| **Always check for leakage** | A perfect score is a red flag, not a victory |
| **Outcome variables aren't features** | `views`, `likes`, `shares` are results, not inputs |
| **Imbalance needs handling** | Default models ignore minority classes |
| **Precision vs. recall is a choice** | Pick based on what a wrong prediction costs you |
| **Realistic > impressive** | An 82% model you can trust beats a 100% model you can't deploy |